<h1>Estudio de Cómics de Marvel</h1>

In [2]:
import csv
import networkx as nx

/opt/sagemath-9.1/local/lib/python3.7/site-packages/scipy/sparse/sputils.py:23: DeprecationWarning: `np.typeDict` is a deprecated alias for `np.sctypeDict`.
  supported_dtypes = [np.typeDict[x] for x in supported_dtypes]
/opt/sagemath-9.1/local/lib/python3.7/site-packages/scipy/linalg/__init__.py:212: DeprecationWarning: The module numpy.dual is deprecated.  Instead of using dual, use the functions directly from numpy or scipy.
  from numpy.dual import register_func
/opt/sagemath-9.1/local/lib/python3.7/site-packages/scipy/special/orthogonal.py:81: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/

In [3]:
G=Graph(weighted=True)
csv_path='../data/processed/marvel/marvel_edges_clean.csv'
export_path="../data/processed/gephi_exports/marvel.gexf"

nodes=set()
edges=set()


with open(csv_path,mode='r') as f:
    lines=csv.reader(f)
    next(lines) #Skip headers
    for l in lines:
        source:str=l[0]
        target:str=l[1]
        weight:int=int(l[2])
        
        nodes.add(source)
        nodes.add(target)
        edges.add((source,target,weight))
        

G.add_vertices(nodes)
G.add_edges(edges)

Gx=G.networkx_graph()
#G.plot(edge_labels=True)
nx.write_gexf(Gx, export_path)
Gx = nx.read_gexf(export_path)
#plot(Graph(Gx))

            
            
    


<h1>Análisis</h1>

<h4>Grado y grado ponderado</h4>
<p>Con quién habla cada personaje y cuanto</p>


In [4]:
grado = dict(Gx.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('Captain America', 258), ('Wolverine / Logan', 220), ('Spider-man / Peter Parker', 218), ('Thing / Benjamin J. Gr', 214), ('Vision', 213), ('Beast / Henry &hank& P', 205), ('Thor / Dr. Donald Blak', 205), ('Iron Man / Tony Stark', 203), ('Mr. Fantastic / Reed R', 198), ('Cyclops / Scott Summer', 197)]


In [6]:
grado_pond = dict(Gx.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('Captain America', 11160.0), ('Spider-man / Peter Parker', 8295.0), ('Thor / Dr. Donald Blak', 8244.0), ('Thing / Benjamin J. Gr', 7907.0), ('Iron Man / Tony Stark', 7804.0), ('Human Torch / Johnny S', 7532.0), ('Vision', 7356.0), ('Scarlet Witch / Wanda', 7303.0), ('Wolverine / Logan', 7193.0), ('Mr. Fantastic / Reed R', 7151.0)]


<h4>Intermediación(betweenes)</h4>
<p>Quien actúa de puente entre grupos </p>

In [8]:
inter = nx.betweenness_centrality(Gx)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('Captain America', 0.06712415218882695), ('Spider-man / Peter Parker', 0.061808932202990254), ('Wolverine / Logan', 0.03548708687569777), ('Beast / Henry &hank& P', 0.031203045357404032), ('Thor / Dr. Donald Blak', 0.028647589789500418), ('Vision', 0.02862430117877163), ('Thing / Benjamin J. Gr', 0.02503653886715355), ('Iron Man / Tony Stark', 0.024893411682719652), ('Cyclops / Scott Summer', 0.022347566374259616), ('Mr. Fantastic / Reed R', 0.021072080826848166)]


<p>La intermediación de un personaje v es la fracción de caminos mínimos entre todos los pares de personajes que pasan por v</p>
<p>Que personajes como Zeus tengan un betweenes de 0 implica que no está en nigún camino mínimo entre dos personajes</p>

<h4>Autovector (eigenvector)</h4>
<p>Quién está conectado con los personajes importantes</p>

In [10]:
auto = nx.eigenvector_centrality(Gx, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('Captain America', 0.28045040108548486), ('Thing / Benjamin J. Gr', 0.24203312284419842), ('Human Torch / Johnny S', 0.23768773385532366), ('Mr. Fantastic / Reed R', 0.2337620475242241), ('Iron Man / Tony Stark', 0.22995062964339794), ('Invisible Woman / Sue', 0.22505409426872905), ('Scarlet Witch / Wanda', 0.21871075469914433), ('Vision', 0.215555561006387), ('Thor / Dr. Donald Blak', 0.20326609610473792), ('Wasp / Janet Van Dyne', 0.202741251851418)]


<p>Valor alto: el personaje está conectado con otros que también tienen valores altos. Es parte del núcleo de la red.</p>
<p>Valor bajo: está en la periferia, o conectado con personajes periféricos </p>

<h4>Cercanía (closeness)</h4>
<p> quién llega más rápido a todos los demás.</p>

In [12]:
cerc = nx.closeness_centrality(Gx)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('Captain America', 0.8274111675126904), ('Wolverine / Logan', 0.7546296296296297), ('Spider-man / Peter Parker', 0.7511520737327189), ('Thing / Benjamin J. Gr', 0.7442922374429224), ('Vision', 0.7425968109339408), ('Beast / Henry &hank& P', 0.7293064876957495), ('Thor / Dr. Donald Blak', 0.7293064876957495), ('Iron Man / Tony Stark', 0.7260579064587973), ('Mr. Fantastic / Reed R', 0.7180616740088106), ('Cyclops / Scott Summer', 0.7164835164835165)]


<h4>Número de nodos, aristas, densidad y componentes conexas.</h4>

In [13]:
print(
    f"nodos: {Gx.number_of_nodes()} aristas: {Gx.number_of_edges()}"
    f" densidad: {nx.density(Gx)}"
)
print(f"componentes conexas: {nx.number_connected_components(Gx)}")

nodos: 327 aristas: 9891 densidad: 0.1855687510553273
componentes conexas: 1


<h4>Comunidades</h4>
<p>¿salen grupos coherentes con la trama (X-man, Gotham-City, UCM...)?</p>

In [15]:
comunidades = nx.community.greedy_modularity_communities(Gx, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

168
['Urich, Ben', 'Mercado, Joy', 'Black Cat / Felicia Ha', 'Cushing, Kate', 'Tombstone / Lonnie Tho', 'Punisher Ii / Frank Ca', 'Hammerhead', 'Robertson, Randy', 'Katzenberg, Nick', 'Kingpin / Wilson Fisk', 'Arranger / ', 'Grant, Gloria Glory', 'Lubenski, Nate', 'Hobgoblin V / Jason Ph', 'Leeds, Ned']
['Klaw / Ulysses Klaw', 'Ultron', 'Volcana / Marsha Rosen', 'Thunderball / Dr. Elio', 'Lizard / Dr. Curtis Co', 'Bulldozer / Henry Camp', 'Molecule Man / Owen Re', 'Beyonder', 'Absorbing Man / Carl C', 'Piledriver Ii / Brian', 'Titania Ii / Mary Skee', 'Wrecker Iii / Dirk Gar']
['Bridge, George Washi', 'Shatterstar Ii / Gavee', 'Deadpool / Jack / Wade W', 'Juggernaut / Cain Mark', 'Domino Iii / Beatrice / ', 'Toad / Mortimer Toynbe', 'Siryn / Theresa Rourke', 'Stryfe', 'Sauron / Dr. Karl Lyko', 'Feral / Maria Callasan', 'Zero']
['Diamondback Ii / Rache', 'Black Mamba / Tanya Se', "O'brien, Michael", 'Stankowicz, Fabian', 'Asp Ii / Cleo', 'Jameson, Col. John', 'Red Skull / Johann Sch', '

<h4>Camino medio y diámetro (grados de separación)</h4>

In [17]:
print(nx.average_shortest_path_length(Gx))
print(nx.diameter(Gx))

1.873004258831917
3


<p>El diámetro es la mayor distancia que hay entre dos personajes cualesquiera de la red</p>

<p>El camino medio es la distancia media entre todos los pares de personajes</p>

<h4>Resultados</h4>


<h2>Resultados del grafo general (Marvel)</h2>

<p><b>Estructura.</b> La red tiene 327 personajes y 9891 aristas, con una densidad de 0,186. Es un grafo conexo (una sola componente), con un camino medio de 1,87 y un diámetro de 3. Es una red muy compacta: cualquier par de personajes está, como mucho, a tres saltos, y de media a menos de dos. El grado medio ronda los 60 contactos por personaje (2·9891/327 ≈ 60,5). Los pesos de las aristas (número de apariciones conjuntas) van de 5 a 744.</p>

<p><b>Ranking de centralidades (top 5).</b> Sin pesos:</p>

<table border="1" style="border-collapse: collapse;">
  <thead>
    <tr>
      <th>Puesto</th>
      <th>Grado</th>
      <th>Cercanía</th>
      <th>Intermediación</th>
      <th>Autovector</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>1</td><td>Captain America (258)</td><td>Captain America (0,827)</td><td>Captain America (0,067)</td><td>Captain America (0,139)</td></tr>
    <tr><td>2</td><td>Wolverine (220)</td><td>Wolverine (0,755)</td><td>Spider-man (0,062)</td><td>Wolverine (0,131)</td></tr>
    <tr><td>3</td><td>Spider-man (218)</td><td>Spider-man (0,751)</td><td>Wolverine (0,035)</td><td>Thing (0,130)</td></tr>
    <tr><td>4</td><td>Thing (214)</td><td>Thing (0,744)</td><td>Beast (0,031)</td><td>Vision (0,127)</td></tr>
    <tr><td>5</td><td>Vision (213)</td><td>Vision (0,743)</td><td>Thor (0,029)</td><td>Beast (0,126)</td></tr>
  </tbody>
</table>

<p>Con pesos (apariciones conjuntas; para cercanía e intermediación la distancia de cada arista es 1/peso):</p>

<table border="1" style="border-collapse: collapse;">
  <thead>
    <tr>
      <th>Puesto</th>
      <th>Grado pond.</th>
      <th>Cercanía pond.</th>
      <th>Intermediación pond.</th>
      <th>Autovector pond.</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>1</td><td>Captain America (11160)</td><td>Captain America (30,64)</td><td>Captain America (0,532)</td><td>Captain America (0,281)</td></tr>
    <tr><td>2</td><td>Spider-man (8295)</td><td>Iron Man (29,40)</td><td>Beast (0,293)</td><td>Thing (0,242)</td></tr>
    <tr><td>3</td><td>Thor (8244)</td><td>Scarlet Witch (29,24)</td><td>Spider-man (0,200)</td><td>Human Torch (0,238)</td></tr>
    <tr><td>4</td><td>Thing (7907)</td><td>Vision (29,23)</td><td>Wolverine (0,186)</td><td>Mr. Fantastic (0,234)</td></tr>
    <tr><td>5</td><td>Iron Man (7804)</td><td>Beast (28,95)</td><td>Cyclops (0,158)</td><td>Iron Man (0,230)</td></tr>
  </tbody>
</table>

<p><i>Nota: Thor y Beast empatan en grado (205, puestos 6.º y 7.º) y también en cercanía (0,7293). Thor y Vision están prácticamente empatados en intermediación (0,02865 frente a 0,02862). Los valores ponderados de cercanía no son comparables con los no ponderados: al usar distancias 1/peso, que son muy pequeñas, la escala cambia.</i></p>

<p><b>El protagonista del universo.</b> Captain America es el primero en las ocho métricas calculadas (grado, grado ponderado, cercanía, cercanía ponderada, intermediación, intermediación ponderada, autovector y autovector ponderado), así que por estructura es el centro de la red: es el que más contactos tiene (258 de un máximo posible de 326), el que más apariciones conjuntas acumula (11160, un 35 % más que Spider-man, que es el segundo con 8295) y el que está más cerca de todos los demás (cercanía 0,827, muy por encima del segundo, 0,755). A diferencia de la saga de Star Wars, aquí el resultado casi no depende de la métrica: un mismo personaje lidera todas. Donde más se reduce su ventaja es en la cercanía ponderada (30,64 frente a 29,40 de Iron Man).</p>

<p><b>Grado frente a grado ponderado.</b> El grado cuenta con cuántos personajes coincide cada uno; el grado ponderado, cuántas veces coincide con ellos. Los grados de la cabeza de la lista son muy parecidos entre sí (del 5.º al 10.º puesto solo van de 213 a 197 contactos), por lo que el grado separa poco y el grado ponderado es la métrica que realmente distingue a los protagonistas. Thor es sexto en grado (205) pero tercero en grado ponderado (8244), unas 40 apariciones por contacto, solo por detrás de Captain America (≈43) y por encima de Spider-man (≈38), Iron Man (≈38) o Thing (≈37): coincide con relativamente pocos personajes, pero muchas veces con ellos. El caso contrario es Wolverine, segundo en grado (220) pero noveno en grado ponderado (7193, unas 33 apariciones por contacto); Vision baja del 5.º puesto en grado al 7.º en ponderado, y Beast (7.º en grado) y Cyclops (10.º) quedan fuera del top 10 ponderado: tienen muchos contactos, pero repartidos en pocas apariciones con cada uno. A la inversa, Human Torch (6.º) y Scarlet Witch (8.º) entran en el top 10 ponderado sin estar en el de grado, lo que indica pocos contactos, pero muy repetidos.</p>

<p><b>Autovector.</b> Sin pesos, el núcleo lo forman Captain America (0,139), Wolverine (0,131), Thing (0,130), Vision (0,127) y Beast (0,126), seguidos de Spider-man, Human Torch, Invisible Woman, Cyclops y Scarlet Witch. Los valores del top 10 son muy parecidos entre sí (de 0,139 a 0,122), lo que refleja una red densa sin una periferia clara entre los personajes principales. Con pesos el orden cambia bastante: Captain America sigue primero (0,281), pero le siguen Thing (0,242), Human Torch (0,238), Mr. Fantastic (0,234) e Invisible Woman (0,225, sexta), es decir, el peso de las apariciones conjuntas favorece a los Cuatro Fantásticos, que coinciden muchísimo entre ellos, y a los Vengadores (Iron Man, Scarlet Witch, Vision, Thor y Wasp). Wolverine, Spider-man, Beast y Cyclops, presentes en el top 10 sin pesos, desaparecen del top 10 ponderado.</p>

<p><b>Cercanía.</b> Sin pesos, Captain America (0,827) está claramente por delante del resto, con un grupo de cola muy compacto (de 0,755 a 0,717 en los puestos 2 a 10), porque con diámetro 3 casi todos llegan a casi todos en uno o dos saltos. Con pesos, el ranking pasa a estar dominado por los Vengadores: Iron Man, Scarlet Witch, Vision, Thor, Wasp, Hawk y Ant-man, además de Beast y Thing. Wolverine y Spider-man, segundo y tercero sin pesos, quedan fuera del top 10: son muy accesibles por número de contactos, pero no tan "cercanos" cuando se valora la frecuencia de las apariciones conjuntas.</p>

<p><b>Los personajes que unen grupos.</b> Los cinco primeros en intermediación (Captain America, Spider-man, Wolverine, Beast y Thor) son personajes que pertenecen o han pertenecido a varios equipos o entornos a la vez (interpretación basada en el conocimiento del universo Marvel, no en la partición en comunidades calculada). Los valores son bajos (el máximo es 0,067) porque con diámetro 3 hay muchos caminos mínimos alternativos y ningún personaje concentra el tráfico. Con pesos, Captain America destaca mucho más (0,532 frente a 0,293 de Beast y 0,200 de Spider-man), porque al preferir las aristas con más apariciones conjuntas los caminos mínimos se concentran en menos personajes. Además, Cyclops sube al quinto puesto (0,158) y entran Hulk (8.º, 0,067) y Cannonball (9.º, 0,059), mientras que Vision sale del top 10.</p>

<p><b>Comunidades.</b> El algoritmo de modularidad voraz (<code>greedy_modularity_communities</code> con pesos) devuelve 168 comunidades, un resultado poco útil: 124 son de un único personaje, solo 12 tienen 6 o más y la mayor tiene 15 personajes. Todos los personajes centrales (Captain America, Wolverine, Spider-man, Thing, Vision, Beast, Thor, Iron Man, Mr. Fantastic, Cyclops, Human Torch, Scarlet Witch) quedan como comunidades de un solo nodo, así que no se puede identificar la estructura en grandes equipos (X-Men, Vengadores, Cuatro Fantásticos...) con esta partición. Lo que sí se recupera son grupos pequeños y coherentes con la trama:</p>

<table border="1" style="border-collapse: collapse;">
  <thead>
    <tr>
      <th>Tamaño</th>
      <th>Personajes</th>
      <th>Interpretación</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>15</td><td>Kingpin, Black Cat, Hammerhead, Tombstone, Ben Urich, Randy Robertson, Joy Mercado, Ned Leeds...</td><td>Entorno urbano de Spider-man y Daredevil (Kingpin y Daily Bugle)</td></tr>
    <tr><td>12</td><td>Klaw, Ultron, Molecule Man, Absorbing Man, Lizard, Titania, Thunderball, Wrecker, Piledriver...</td><td>Supervillanos (Wrecking Crew y otros)</td></tr>
    <tr><td>11</td><td>Deadpool, Juggernaut, Toad, Siryn, Stryfe, Sauron, Feral, Shatterstar...</td><td>Mutantes del entorno de X-Force y X-Men</td></tr>
    <tr><td>8</td><td>Power Man (Erik Josten), Beetle, Moonstone, Screaming Mimi, Fixer, Triathlon, Gyrich...</td><td>Thunderbolts</td></tr>
    <tr><td>8</td><td>Gladiator, Deathbird, Lilandra, Corsair, Ch'od, Hepzibah, Raza, Karma</td><td>Imperio Shi'ar</td></tr>
    <tr><td>7</td><td>Blob, Pyro, Avalanche, Spiral, Destiny, Longshot, Madelyne Pryor</td><td>Hermandad de Mutantes y Fuerza Libertadora</td></tr>
    <tr><td>6</td><td>Makkari, Thena, Sersi, Ikaris, Forgotten One, Ghaur</td><td>Eternos y Desviantes</td></tr>
  </tbody>
</table>

<p>Es decir, el algoritmo detecta bien los grupos pequeños y muy cohesionados (los que se relacionan casi solo entre sí), pero no agrupa a los grandes personajes que conectan con todos los equipos, que son precisamente los que más pesan en la red. Para identificar los grandes equipos habría que repetir la detección con otro algoritmo (por ejemplo Louvain) y comprobar el número de comunidades y la modularidad.</p>

<p><b>Síntesis.</b> Captain America es el protagonista estructural del universo: lidera las ocho métricas de centralidad. Wolverine y Spider-man son los siguientes en grado, cercanía e intermediación sin pesos; con pesos, Spider-man y Thor encabezan el grado ponderado tras Captain America, Beast el segundo puesto en intermediación ponderada y los Vengadores y los Cuatro Fantásticos destacan en cercanía y autovector ponderados. La red es muy densa (0,186), conexa, con diámetro 3 y camino medio 1,87. La detección de comunidades con modularidad voraz solo aísla grupos pequeños y deja a los personajes principales como nodos sueltos, por lo que no permite extraer conclusiones sobre los grandes equipos.</p>


<h2>Exportar grafo como html</h2>

In [18]:
import os
import re
import math
import json
import numpy as np
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------
# Número máximo de nodos que se dibujan en el HTML (los de mayor grado ponderado).
# Con miles de personajes el HTML y el layout se vuelven inusables; 250 es un buen punto de partida.
MAX_NODOS_VISUALIZACION = 327

# Wiki de Fandom de la que se sacan las fotos de los personajes
FANDOM_WIKI = "marvel.fandom.com"

os.makedirs("grafos_interactivos", exist_ok=True)
os.makedirs("gephi_preconfigurados", exist_ok=True)


def serializador_sage(obj):
    if hasattr(obj, "item"):  # Tipos de NumPy (np.int64, np.float64)
        return obj.item()
    try:
        return int(obj) if float(obj).is_integer() else float(obj)
    except (TypeError, ValueError):
        return str(obj)


# ---------------------------------------------------------------------------
# OBTENER EL GRAFO DEL NOTEBOOK
# ---------------------------------------------------------------------------
# Si en tu notebook el grafo se llama G (grafo de Sage o de networkx), esto lo convierte
# a networkx. Si lo cargas desde el CSV, ajusta los nombres de columna.
def a_networkx(G):
    """Convierte un Graph de Sage a networkx (si ya es networkx lo deja igual)."""
    if hasattr(G, "networkx_graph"):
        G = G.networkx_graph()
    return nx.Graph(G)


def cargar_desde_csv(ruta="marvel_edges_clean.csv", col_u="source", col_v="target", col_w="weight"):
    import pandas as pd
    df = pd.read_csv(ruta)
    G = nx.Graph()
    for u, v, w in zip(df[col_u], df[col_v], df[col_w] if col_w in df.columns else [1] * len(df)):
        if u != v:
            G.add_edge(str(u), str(v), weight=float(w))
    return G


PALETA_COMUNIDADES = [
    "#E63946", "#1D3557", "#2A9D8F", "#F4A261",
    "#8338EC", "#06D6A0", "#FF006E", "#3A86FF",
    "#FB5607", "#8AC926", "#6A4C93", "#1982C4",
]
COLOR_RESTO = "#90A4AE"


def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))


def nombre_para_busqueda(nombre):
    """
    En los datasets de Marvel los nombres suelen venir como 'SPIDER-MAN/PETER PARKER'
    o 'CAPTAIN AMERICA'. Para buscar la foto nos quedamos con el alias y lo pasamos a
    formato título.
    """
    n = str(nombre).split("/")[0]
    n = re.sub(r"\(.*?\)", "", n).replace("_", " ").strip()
    return n.title() if n.isupper() else n


# ---------------------------------------------------------------------------
# LAYOUT
# ---------------------------------------------------------------------------
def separar_nodos_cercanos(pos, dist_minima=0.065, iteraciones=40):
    """Repulsión suave entre pares de nodos que estén más cerca que dist_minima (vectorizada)."""
    nodos = list(pos.keys())
    coords = np.array([pos[n] for n in nodos], dtype=float)
    if len(nodos) < 2:
        return pos

    for _ in range(iteraciones):
        delta = coords[:, None, :] - coords[None, :, :]
        dist = np.linalg.norm(delta, axis=2)
        np.fill_diagonal(dist, np.inf)
        cerca = dist < dist_minima
        if not cerca.any():
            break
        dist_seg = np.where(dist < 1e-6, 1e-6, dist)
        direccion = delta / dist_seg[:, :, None]
        empuje = 0.5 * (dist_minima - dist_seg)[:, :, None] * direccion
        empuje = np.where(cerca[:, :, None], empuje, 0.0)
        coords += empuje.sum(axis=1)

    return {nodos[i]: coords[i] for i in range(len(nodos))}


def calcular_layout_compacto(G, nodo_a_comunidad, n_coms_principales, seed=42):
    rng = np.random.RandomState(int(seed))
    componentes = sorted(nx.connected_components(G), key=len, reverse=True)

    G_main = G.subgraph(componentes[0]).copy()
    n_main = G_main.number_of_nodes()
    k_optimo = float(4.2 / math.sqrt(n_main)) if n_main < 200 else float(5.5 / math.sqrt(n_main))

    pos_init = {}
    for n in G_main.nodes():
        c_idx = nodo_a_comunidad[n] - 1
        if c_idx < n_coms_principales:
            angulo = float(2.0 * math.pi * c_idx / n_coms_principales)
            cx, cy = float(0.75 * math.cos(angulo)), float(0.75 * math.sin(angulo))
        else:
            cx, cy = 0.0, 0.0
        pos_init[n] = np.array([cx + rng.normal(0.0, 0.22), cy + rng.normal(0.0, 0.22)], dtype=float)

    pos = nx.spring_layout(G_main, pos=pos_init, k=k_optimo, weight="weight",
                           iterations=300, seed=int(seed))

    coords = np.array(list(pos.values()))
    min_xy, max_xy = coords.min(axis=0), coords.max(axis=0)
    span = np.maximum(max_xy - min_xy, 1e-6)
    for n in pos:
        pos[n] = 2.0 * (pos[n] - min_xy) / span - 1.0

    pos = separar_nodos_cercanos(pos, dist_minima=0.065, iteraciones=50)

    # Componentes secundarias (islas) en el borde inferior derecho
    offset_x, offset_y = 1.25, -0.90
    for comp in componentes[1:]:
        subG = G.subgraph(comp)
        sub_pos = nx.spring_layout(subG, k=0.22, seed=int(seed))
        for n, xy in sub_pos.items():
            pos[n] = np.array([offset_x + float(xy[0]) * 0.10, offset_y + float(xy[1]) * 0.10])
        offset_y += 0.28

    return pos


# ---------------------------------------------------------------------------
# GENERACIÓN DEL HTML + GEXF
# ---------------------------------------------------------------------------
def generar_grafo_interactivo(G, nombre_archivo, titulo,
                              metrica_tamano="grado_ponderado", seed=42,
                              max_nodos=MAX_NODOS_VISUALIZACION):
    seed = int(seed)
    G_vis = a_networkx(G)
    G_vis = nx.relabel_nodes(G_vis, {n: str(n) for n in G_vis.nodes()})
    n_total_original = G_vis.number_of_nodes()

    # Quedarse con los nodos más relevantes si el grafo es muy grande
    if max_nodos and n_total_original > max_nodos:
        gp = dict(G_vis.degree(weight="weight"))
        top = sorted(gp, key=gp.get, reverse=True)[:max_nodos]
        G_vis = G_vis.subgraph(top).copy()
    G_vis.remove_nodes_from(list(nx.isolates(G_vis)))

    n_nodos = int(G_vis.number_of_nodes())
    n_aristas = int(G_vis.number_of_edges())

    # 1. Métricas
    grado = dict(G_vis.degree())
    grado_pond = dict(G_vis.degree(weight="weight"))
    intermediacion = nx.betweenness_centrality(G_vis, weight=None)
    try:
        autovector = nx.eigenvector_centrality(G_vis, weight="weight", max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        autovector = nx.eigenvector_centrality_numpy(G_vis, weight="weight")

    metricas = {
        "grado": grado,
        "grado_ponderado": grado_pond,
        "intermediacion": intermediacion,
        "autovector": autovector,
    }
    valores_metrica = metricas[metrica_tamano]

    # 2. Comunidades
    comunidades = sorted(greedy_modularity_communities(G_vis, weight="weight"),
                         key=len, reverse=True)

    nodo_a_comunidad, nodo_a_color, leyenda_items = {}, {}, []
    for idx, com in enumerate(comunidades):
        color = PALETA_COMUNIDADES[idx] if idx < len(PALETA_COMUNIDADES) else COLOR_RESTO
        lider = max(com, key=lambda n: grado_pond[n])
        if idx < 10 and len(com) >= 3:
            leyenda_items.append((color, f"Com. {idx + 1}: {nombre_para_busqueda(lider)} ({len(com)} pers.)"))
        for nodo in com:
            nodo_a_comunidad[nodo] = int(idx + 1)
            nodo_a_color[nodo] = color

    # 3. Layout
    pos = calcular_layout_compacto(G_vis, nodo_a_comunidad, min(len(comunidades), 12), seed=seed)

    # 4. Nodos y aristas para vis-network
    vals = np.array([float(valores_metrica[n]) for n in G_vis.nodes()], dtype=float)
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    escala_lienzo = 1800.0

    nodos_json = []
    for idx_n, n in enumerate(G_vis.nodes()):
        relevancia = float(vals_norm[idx_n])
        tamano = float(20.0 + 54.0 * (relevancia ** 0.75))
        font_size = int(22 + 32 * (relevancia ** 0.7))
        x, y = float(pos[n][0] * escala_lienzo), float(pos[n][1] * escala_lienzo)
        etiqueta = nombre_para_busqueda(n)

        tooltip = (
            f"<b>{n}</b> (Comunidad {nodo_a_comunidad[n]})<br>"
            f"Grado: {int(grado[n])}<br>"
            f"Grado ponderado: {int(grado_pond[n])}<br>"
            f"Intermediación: {float(intermediacion[n]):.4f}<br>"
            f"Autovector: {float(autovector[n]):.4f}"
        )

        nodos_json.append({
            "id": str(n),
            "label": etiqueta,
            "busqueda": etiqueta,
            "x": x, "y": y, "baseX": x, "baseY": y,
            "baseFontSize": font_size,
            "size": tamano,
            "shape": "dot",
            "borderWidth": 5,
            "color": {
                "background": "#B0BEC5",
                "border": nodo_a_color[n],
                "highlight": {"background": "#CFD8DC", "border": nodo_a_color[n]},
            },
            "font": {
                "size": font_size, "color": "#111111", "face": "Arial",
                "bold": bool(relevancia > 0.35),
                "strokeWidth": 5, "strokeColor": "#FFFFFF",
            },
            "title": tooltip,
        })

    pesos = [float(d.get("weight", 1.0)) for _, _, d in G_vis.edges(data=True)]
    p_max = max(pesos) if pesos else 1.0
    aristas_json = []
    for u, v, d in G_vis.edges(data=True):
        w = float(d.get("weight", 1.0))
        aristas_json.append({
            "from": str(u), "to": str(v), "value": w,
            "width": float(0.8 + 4.2 * (w / p_max)),
            "title": f"{u} ↔ {v} (Peso: {int(w)})",
            "color": {"color": "rgba(120, 144, 156, 0.28)", "highlight": "#E63946"},
        })

    leyenda_html = "".join(
        f'<div style="display:flex;align-items:center;margin-bottom:4px;">'
        f'<span style="width:12px;height:12px;background:{c};display:inline-block;margin-right:6px;border-radius:50%;"></span>'
        f'<span>{lbl}</span></div>'
        for c, lbl in leyenda_items
    )

    nota_recorte = (f" (top {n_nodos} de {n_total_original})"
                    if n_nodos < n_total_original else "")

    html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>{titulo}</title>
  <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <style>
    body {{ margin: 0; padding: 0; font-family: Arial, sans-serif; background: #FAFAFA; overflow: hidden; }}
    #header {{
      position: absolute; top: 12px; left: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 12px 16px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); max-width: 480px;
    }}
    #header h2 {{ margin: 0 0 4px 0; font-size: 18px; color: #1A1A1A; }}
    #header p {{ margin: 0; font-size: 12px; color: #555; }}
    #legend {{
      position: absolute; bottom: 16px; left: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 10px 14px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 12px;
    }}
    #controls {{
      position: absolute; top: 12px; right: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 10px 12px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 12px;
      display: flex; flex-direction: column; gap: 8px;
    }}
    .btn-row {{ display: flex; gap: 6px; }}
    .slider-row {{ display: flex; align-items: center; justify-content: space-between; gap: 8px; color: #333; font-weight: bold; }}
    button {{
      padding: 6px 10px; cursor: pointer; border: 1px solid #CCC;
      background: #FFF; border-radius: 4px; font-weight: bold;
    }}
    button:hover {{ background: #EEE; }}
    #mynetwork {{ width: 100vw; height: 100vh; }}
  </style>
</head>
<body>
  <div id="header">
    <h2>{titulo}</h2>
    <p>Nodos: {n_nodos}{nota_recorte} | Aristas: {n_aristas} | Tamaño: {metrica_tamano} | Borde = Comunidad</p>
  </div>
  <div id="controls">
    <div class="btn-row">
      <button onclick="network.fit({{animation: true}})">Centrar grafo</button>
      <button id="btnPhysics" onclick="togglePhysics()">Activar física</button>
    </div>
    <div class="slider-row">
      <label for="spreadSlider">Separación:</label>
      <input type="range" id="spreadSlider" min="0.5" max="50" step="0.1" value="1.0" oninput="escalarDisposicion(this.value)">
      <span id="spreadVal">1.0x</span>
    </div>
    <div class="slider-row">
      <label for="fontSlider">Tamaño texto:</label>
      <input type="range" id="fontSlider" min="0.5" max="2.5" step="0.1" value="1.0" oninput="escalarTexto(this.value)">
      <span id="fontVal">1.0x</span>
    </div>
  </div>
  <div id="legend">
    <div style="font-weight:bold;margin-bottom:6px;">Comunidades (color del borde)</div>
    {leyenda_html}
  </div>
  <div id="mynetwork"></div>

  <script type="text/javascript">
    var nodes = new vis.DataSet({json.dumps(nodos_json, default=serializador_sage)});
    var edges = new vis.DataSet({json.dumps(aristas_json, default=serializador_sage)});
    var container = document.getElementById('mynetwork');
    var data = {{ nodes: nodes, edges: edges }};
    var physicsEnabled = false;

    var options = {{
      nodes: {{
        borderWidth: 5,
        shapeProperties: {{ useBorderWithImage: true }},
        scaling: {{ label: {{ enabled: false }} }}
      }},
      edges: {{ smooth: {{ type: 'continuous' }} }},
      physics: {{
        enabled: false,
        solver: 'forceAtlas2Based',
        forceAtlas2Based: {{
          gravitationalConstant: -180, centralGravity: 0.005,
          springLength: 220, springConstant: 0.04, damping: 0.4, avoidOverlap: 1
        }},
        stabilization: {{ enabled: false }}
      }},
      interaction: {{ hover: true, tooltipDelay: 100, navigationButtons: true }}
    }};

    var network = new vis.Network(container, data, options);
    network.once("afterDrawing", function() {{ network.fit(); }});

    // Fotos desde la API de Fandom (Marvel). Si no se encuentra, queda el círculo gris.
    async function cargarFotosPersonajes() {{
      var listaNodos = nodes.get();
      for (var i = 0; i < listaNodos.length; i++) {{
        var n = listaNodos[i];
        try {{
          var url = "https://{FANDOM_WIKI}/api.php?action=query&generator=search&gsrsearch=" +
                    encodeURIComponent(n.busqueda) +
                    "&gsrlimit=4&prop=pageimages&piprop=thumbnail&pithumbsize=250&format=json&origin=*";
          var resp = await fetch(url);
          var json = await resp.json();
          if (json.query && json.query.pages) {{
            var paginas = Object.values(json.query.pages).sort(function(a, b) {{ return a.index - b.index; }});
            var personaje = paginas.find(function(p) {{
              var t = (p.title || "").toLowerCase();
              return p.thumbnail && p.thumbnail.source &&
                     !t.includes("disambiguation") && !t.includes("category:");
            }});
            if (personaje) {{
              nodes.update({{ id: n.id, shape: "circularImage", image: personaje.thumbnail.source }});
            }}
          }}
        }} catch (e) {{}}
      }}
    }}
    cargarFotosPersonajes();

    function togglePhysics() {{
      physicsEnabled = !physicsEnabled;
      document.getElementById('btnPhysics').innerText = physicsEnabled ? 'Pausar física' : 'Activar física';
      network.setOptions({{ physics: {{ enabled: physicsEnabled }} }});
    }}

    function escalarDisposicion(factor) {{
      var f = parseFloat(factor);
      document.getElementById('spreadVal').innerText = f.toFixed(1) + 'x';
      var updates = [];
      nodes.forEach(function(n) {{ updates.push({{ id: n.id, x: n.baseX * f, y: n.baseY * f }}); }});
      nodes.update(updates);
      network.fit({{ animation: false }});
    }}

    function escalarTexto(factor) {{
      var f = parseFloat(factor);
      document.getElementById('fontVal').innerText = f.toFixed(1) + 'x';
      var updates = [];
      nodes.forEach(function(n) {{
        updates.push({{ id: n.id, font: {{ size: Math.round(n.baseFontSize * f) }} }});
      }});
      nodes.update(updates);
    }}
  </script>
</body>
</html>"""

    ruta_html = f"grafos_interactivos/{nombre_archivo}_{metrica_tamano}.html"
    with open(ruta_html, "w", encoding="utf-8") as f:
        f.write(html_content)

    # 6. GEXF para Gephi (con las coordenadas del layout)
    for idx_n, n in enumerate(G_vis.nodes()):
        r, g, b = hex_to_rgb(nodo_a_color[n])
        x, y = float(pos[n][0]), float(pos[n][1])
        G_vis.nodes[n]["label"] = nombre_para_busqueda(n)
        G_vis.nodes[n]["grado"] = int(grado[n])
        G_vis.nodes[n]["grado_ponderado"] = float(grado_pond[n])
        G_vis.nodes[n]["intermediacion"] = float(intermediacion[n])
        G_vis.nodes[n]["autovector"] = float(autovector[n])
        G_vis.nodes[n]["comunidad"] = f"Comunidad_{nodo_a_comunidad[n]}"
        G_vis.nodes[n]["viz"] = {
            "color": {"r": int(r), "g": int(g), "b": int(b), "a": 1.0},
            "size": float(15.0 + 60.0 * (vals_norm[idx_n] ** 0.75)),
            "position": {"x": float(x * escala_lienzo), "y": float(y * escala_lienzo), "z": 0.0},
        }

    for u, v, d in G_vis.edges(data=True):
        if "weight" in d:
            d["weight"] = float(d["weight"])

    ruta_gexf = f"gephi_preconfigurados/{nombre_archivo}_{metrica_tamano}.gexf"
    nx.write_gexf(G_vis, ruta_gexf)
    print(f"[OK] Generado: {ruta_html} y {ruta_gexf}")


# ---------------------------------------------------------------------------
# EJECUCIÓN
# ---------------------------------------------------------------------------
# Usa el grafo que ya tienes en el notebook (aquí se asume que se llama G).
# Si no lo tienes en memoria, descomenta la línea de carga desde el CSV.
# G = cargar_desde_csv("marvel_edges_clean.csv")   # ajusta nombres de columna si hace falta

grafos_estudio = [
    (G, "00_marvel_completo", "Universo Marvel — Red Global de Colaboraciones"),
    # Añade aquí subgrafos si los has construido en el notebook, p. ej.:
    # (G_vengadores, "01_vengadores", "Marvel — Subred de los Vengadores"),
]

for G_actual, archivo, titulo in grafos_estudio:
    generar_grafo_interactivo(G_actual, nombre_archivo=archivo, titulo=titulo,
                              metrica_tamano="grado_ponderado")

# Extra: personajes puente entre comunidades
generar_grafo_interactivo(G, nombre_archivo="00_marvel_completo",
                          titulo="Universo Marvel — Personajes Puente entre Comunidades",
                          metrica_tamano="intermediacion")

/opt/sagemath-9.1/local/lib/python3.7/site-packages/sage/repl/ipython_kernel/__main__.py:97: RuntimeWarning: invalid value encountered in multiply


[OK] Generado: grafos_interactivos/00_marvel_completo_grado_ponderado.html y gephi_preconfigurados/00_marvel_completo_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/00_marvel_completo_intermediacion.html y gephi_preconfigurados/00_marvel_completo_intermediacion.gexf
